# 🚀 Fiuld Engine - ARC-AGI-3 Solver

Pure Rust genetic algorithm solver with ColorRef algebraic generalization and Reflection Loop.

**Engine Specs:**
- Population: 2000 | Generations: 150 | Mutation: 15%
- DSL Actions: Fill, Line, Rect, ColorSwap, Rotate, Flip, Scale, Crop, Pad, MoveObject, Copy
- ColorRef: Exact, Background, Foreground, ForegroundMax
- Reflection: 3-tier patching (Micro-patch → ColorSwap → BlockFill)

In [ ]:
import os
import subprocess
import time
import json

# ============================================================
# 1. Environment Setup
# ============================================================

DATASET_PATH = "/kaggle/input/fiuld-binary-dataset"
EXECUTABLE_NAME = "fiuld"
EXEC_PATH = os.path.join(DATASET_PATH, EXECUTABLE_NAME)
TEST_JSON = "/kaggle/input/arc-prize-2026/test.json"
SUBMISSION_OUTPUT = "/kaggle/working/submission.json"

print("🚀 Preparing Fiuld engine launch sequence...")
print(f"Executable: {EXEC_PATH}")
print(f"Input: {TEST_JSON}")
print(f"Output: {SUBMISSION_OUTPUT}")

# Verify files exist
assert os.path.exists(EXEC_PATH), f"Executable not found: {EXEC_PATH}"
assert os.path.exists(TEST_JSON), f"Test JSON not found: {TEST_JSON}"

# Make executable
os.chmod(EXEC_PATH, 0o755)
print(f"✅ Executable ready: {os.path.getsize(EXEC_PATH)} bytes")

In [ ]:
# ============================================================
# 2. Launch Fiuld Engine
# ============================================================

print("⚔️ Launching Fiuld - Full inference starting...")
start_time = time.time()

try:
    process = subprocess.run(
        [EXEC_PATH, TEST_JSON, SUBMISSION_OUTPUT],
        check=True,
        capture_output=True,
        text=True,
        timeout=32400  # 9 hours
    )
    
    # Print engine logs
    print("--- Engine Output ---")
    print(process.stdout)
    print("--------------------")
    
    if process.stderr:
        print("--- Engine Stderr ---")
        print(process.stderr)
        print("---------------------")
        
except subprocess.CalledProcessError as e:
    print(f"❌ Engine crashed (Exit Code: {e.returncode})")
    print("--- Error Output ---")
    print(e.stdout)
    print(e.stderr)
    print("--------------------")
except subprocess.TimeoutExpired:
    print("⚠️ Engine timed out after 9 hours")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

end_time = time.time()
elapsed_minutes = (end_time - start_time) / 60
print(f"🏁 Inference complete! Total time: {elapsed_minutes:.2f} minutes")

In [ ]:
# ============================================================
# 3. Validation & Submission Check
# ============================================================

if os.path.exists(SUBMISSION_OUTPUT):
    file_size = os.path.getsize(SUBMISSION_OUTPUT)
    print(f"✅ Submission file created: {SUBMISSION_OUTPUT}")
    print(f"   File size: {file_size / 1024:.2f} KB")
    
    # Validate JSON structure
    with open(SUBMISSION_OUTPUT, 'r') as f:
        submission = json.load(f)
    
    print(f"   Total predictions: {len(submission)}")
    print(f"   Keys: {list(submission.keys())[:5]}...")
    
    # Check first prediction format
    first_key = list(submission.keys())[0]
    first_pred = submission[first_key]
    if isinstance(first_pred, list) and len(first_pred) > 0:
        print(f"   Sample output ({first_key}): {len(first_pred)}x{len(first_pred[0])}")
    
    print("\n🎯 All checks passed! Ready for upload.")
else:
    print("⚠️ WARNING: submission.json not found! Engine may have failed.")